# Day 30: Global Health Dataset - Comprehensive EDA & Ensemble ML

## Final Day Special: Maximum Extensive Analysis on Unified Global Health Dataset

This notebook performs comprehensive exploratory data analysis and builds an ensemble voting classifier to predict health outcomes from 22,050 global health records covering 30+ years of data.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


## Section 1: Load and Explore the Dataset

In [2]:
# Load the dataset
df = pd.read_csv('../data/UnifiedDataset.csv')

# Display basic information
print(f"Dataset Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMissing Values (top 15):")
print(df.isnull().sum().sort_values(ascending=False).head(15))
print(f"\nBasic Statistics:")
print(df.describe())

Dataset Shape: (22050, 150)

First few rows:
       Country  Year      Gender  Life Expectancy  Infant Mortality Rate  \
0  Afghanistan  1990  Both sexes           50.331                  120.4   
1  Afghanistan  1990      Female           51.442                  114.2   
2  Afghanistan  1990        Male           49.281                  126.2   
3  Afghanistan  1991  Both sexes           50.999                  116.8   
4  Afghanistan  1991      Female           52.119                  110.7   

   Low CI Value Infant Mortality Rate  High CI Value Infant Mortality Rate  \
0                               111.2                                130.9   
1                               105.1                                124.7   
2                               116.4                                137.5   
3                               108.2                                126.2   
4                               102.1                                120.4   

   Under 5 Mortality Rate  Lo

## Section 2: Data Cleaning and Preprocessing

In [3]:
# Create a copy for processing
df_clean = df.copy()

# Handle missing values - fill with median for numerical columns
print("Handling missing values...")
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Remove duplicate rows
print(f"Duplicate rows before: {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates()
print(f"Duplicate rows after: {df_clean.duplicated().sum()}")

# Create target variable: High vs Low Life Expectancy
life_exp_median = df_clean['Life Expectancy'].median()
df_clean['Life_Expectancy_Category'] = (df_clean['Life Expectancy'] > life_exp_median).astype(int)

print(f"\nLife Expectancy Categories:")
print(df_clean['Life_Expectancy_Category'].value_counts())
print(f"\nCleaned dataset shape: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

Handling missing values...
Duplicate rows before: 0
Duplicate rows after: 0

Life Expectancy Categories:
Life_Expectancy_Category
0    11025
1    11025
Name: count, dtype: int64

Cleaned dataset shape: (22050, 151)
Missing values: 0


## Section 3: Exploratory Data Analysis with Plotly

In [4]:
# 1. Life Expectancy Distribution
fig1 = px.histogram(df_clean, x='Life Expectancy', nbins=50, template='plotly_dark',
                    title='Life Expectancy Distribution Worldwide',
                    labels={'Life Expectancy': 'Years', 'count': 'Frequency'},
                    color_discrete_sequence=['#00CC96'])
fig1.update_layout(showlegend=False, height=500)
fig1.write_html('../viz/life_expectancy_distribution.html')
fig1.show()

# 2. Infant Mortality vs Life Expectancy
fig2 = px.scatter(df_clean, x='Infant Mortality Rate', y='Life Expectancy', 
                  color='Year', template='plotly_dark',
                  title='Infant Mortality vs Life Expectancy Over Time',
                  hover_data=['Country', 'Year'],
                  color_continuous_scale='Viridis')
fig2.update_layout(height=500)
fig2.write_html('../viz/mortality_vs_life_expectancy.html')
fig2.show()

In [5]:
# 3. GDP vs Life Expectancy by Gender
fig3 = px.scatter(df_clean.dropna(subset=['GDP per Capita']), 
                  x='GDP per Capita', y='Life Expectancy',
                  color='Gender', template='plotly_dark',
                  title='GDP per Capita vs Life Expectancy by Gender',
                  size='Total Population', hover_data=['Country', 'Year'],
                  color_discrete_map={'Both sexes': '#636EFA', 'Female': '#EF553B', 'Male': '#00CC96'})
fig3.update_layout(height=500)
fig3.write_html('../viz/gdp_vs_life_expectancy.html')
fig3.show()

# 4. Healthcare Coverage Impact
fig4 = px.box(df_clean.dropna(subset=['Universal Heath Care Coverage']),
              x='Gender', y='Life Expectancy', color='Gender',
              template='plotly_dark',
              title='Life Expectancy by Gender and Healthcare Coverage',
              color_discrete_map={'Both sexes': '#636EFA', 'Female': '#EF553B', 'Male': '#00CC96'})
fig4.update_layout(height=500)
fig4.write_html('../viz/healthcare_impact.html')
fig4.show()

In [6]:
# 5. Temporal Trends in Life Expectancy
yearly_avg = df_clean.groupby('Year')['Life Expectancy'].mean()
fig5 = px.line(x=yearly_avg.index, y=yearly_avg.values, template='plotly_dark',
               title='Global Average Life Expectancy Trend (1990-2021)',
               labels={'x': 'Year', 'y': 'Life Expectancy (Years)'},
               markers=True, color_discrete_sequence=['#00CC96'])
fig5.update_layout(height=500)
fig5.write_html('../viz/life_expectancy_trend.html')
fig5.show()

# 6. Correlation Heatmap of Key Indicators
numeric_subset = df_clean[['Life Expectancy', 'Infant Mortality Rate', 'Under 5 Mortality Rate',
                           'Suicides Rate', 'GDP per Capita', 'Birth Rate', 'Death Rate']].dropna()
corr_matrix = numeric_subset.corr()

fig6 = go.Figure(data=go.Heatmap(z=corr_matrix.values,
                                 x=corr_matrix.columns,
                                 y=corr_matrix.columns,
                                 colorscale='RdBu',
                                 zmid=0,
                                 text=corr_matrix.values,
                                 texttemplate='%.2f',
                                 textfont={"size": 10}))
fig6.update_layout(title='Correlation Heatmap of Key Health Indicators',
                   template='plotly_dark', height=600, width=800)
fig6.write_html('../viz/correlation_heatmap.html')
fig6.show()

## Section 4: Feature Engineering and Selection

In [7]:
# Select relevant features for modeling
feature_columns = [
    'Year', 'Gender', 'Infant Mortality Rate', 'Under 5 Mortality Rate',
    'Suicides Rate', 'Birth Rate', 'Death Rate', 'GDP per Capita',
    'Total Population', 'Doctors', 'Nurses and Midwifes', 'Road Traffic Deaths'
]

# Check which columns exist and have valid datafollowing the conv
available_features = [col for col in feature_columns if col in df_clean.columns]
print(f"Available features: {available_features}")

# Create feature matrix
df_model = df_clean[available_features + ['Life_Expectancy_Category']].copy()
df_model = df_model.dropna()

print(f"Model dataset shape: {df_model.shape}")
print(f"Feature statistics:")
print(df_model.describe())

# Encode categorical variables
le_gender = LabelEncoder()
df_model['Gender_encoded'] = le_gender.fit_transform(df_model['Gender'])

# Drop original categorical columns
X = df_model.drop(['Gender', 'Life_Expectancy_Category'], axis=1)
y = df_model['Life_Expectancy_Category']

print(f"\nFeatures shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())
print(f"Feature columns: {list(X.columns)}")

Available features: ['Year', 'Gender', 'Infant Mortality Rate', 'Under 5 Mortality Rate', 'Suicides Rate', 'Birth Rate', 'Death Rate', 'GDP per Capita', 'Total Population', 'Doctors', 'Nurses and Midwifes', 'Road Traffic Deaths']
Model dataset shape: (22050, 13)
Feature statistics:
               Year  Infant Mortality Rate  Under 5 Mortality Rate  \
count  22050.000000           22050.000000            22050.000000   
mean    2004.500000              29.487496               40.505883   
std        8.655638              26.838320               44.716966   
min     1990.000000               1.420000                1.680000   
25%     1997.000000              14.412500               16.850000   
50%     2004.500000              21.410000               25.280000   
75%     2012.000000              32.287500               40.327500   
max     2019.000000             189.200000              331.100000   

       Suicides Rate    Birth Rate    Death Rate  GDP per Capita  \
count   22050.0000

## Section 5: Train-Test Split and Scaling

In [8]:
# Split data into training and testing sets (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"Training target distribution:")
print(y_train.value_counts())
print(f"Testing target distribution:")
print(y_test.value_counts())

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nFeatures scaled successfully!")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

Training set size: (17640, 12)
Testing set size: (4410, 12)
Training target distribution:
Life_Expectancy_Category
1    8820
0    8820
Name: count, dtype: int64
Testing target distribution:
Life_Expectancy_Category
1    2205
0    2205
Name: count, dtype: int64

Features scaled successfully!
X_train_scaled shape: (17640, 12)
X_test_scaled shape: (4410, 12)


## Section 6: Build Individual ML Models

In [9]:
# 1. Random Forest Classifier
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"Random Forest Accuracy: {rf_acc:.4f}")

# 2. Gradient Boosting Classifier
print("Training Gradient Boosting...")
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_acc = accuracy_score(y_test, gb_pred)
print(f"Gradient Boosting Accuracy: {gb_acc:.4f}")

# 3. Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)
print(f"Logistic Regression Accuracy: {lr_acc:.4f}")

# 4. K-Nearest Neighbors
print("Training K-Nearest Neighbors...")
knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)
knn_acc = accuracy_score(y_test, knn_pred)
print(f"K-Nearest Neighbors Accuracy: {knn_acc:.4f}")

print("\nIndividual models trained successfully!")

Training Random Forest...
Random Forest Accuracy: 0.9050
Training Gradient Boosting...
Gradient Boosting Accuracy: 0.8973
Training Logistic Regression...
Logistic Regression Accuracy: 0.8472
Training K-Nearest Neighbors...
K-Nearest Neighbors Accuracy: 0.8800

Individual models trained successfully!


## Section 7: Create Voting Classifier Ensemble

In [12]:
# Create Voting Classifier with soft voting
print("Creating Voting Classifier Ensemble...")
voting_clf_soft = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('lr', lr_model),
        ('knn', knn_model)
    ],
    voting='soft'
)

# Fit the voting classifier (it will use the already-trained estimators)
voting_clf_soft.fit(X_train_scaled, y_train)

# Make predictions using the ensemble
ensemble_pred_soft = voting_clf_soft.predict(X_test_scaled)
ensemble_pred_proba = voting_clf_soft.predict_proba(X_test_scaled)
ensemble_acc_soft = accuracy_score(y_test, ensemble_pred_soft)

print(f"\nVoting Classifier (Soft) Accuracy: {ensemble_acc_soft:.4f}")

# Calculate F1 scores for all models
print(f"\nF1-Score Comparison:")
print(f"Random Forest: {f1_score(y_test, rf_pred):.4f}")
print(f"Gradient Boosting: {f1_score(y_test, gb_pred):.4f}")
print(f"Logistic Regression: {f1_score(y_test, lr_pred):.4f}")
print(f"K-Nearest Neighbors: {f1_score(y_test, knn_pred):.4f}")
print(f"Voting Classifier: {f1_score(y_test, ensemble_pred_soft):.4f}")

# Store best model (using ensemble)
best_model = voting_clf_soft
best_predictions = ensemble_pred_soft

Creating Voting Classifier Ensemble...

Voting Classifier (Soft) Accuracy: 0.8993

F1-Score Comparison:
Random Forest: 0.9059
Gradient Boosting: 0.8987
Logistic Regression: 0.8491
K-Nearest Neighbors: 0.8824
Voting Classifier: 0.9014


## Section 8: Model Evaluation and Comparison

In [14]:
# Confusion Matrix for Ensemble
cm = confusion_matrix(y_test, ensemble_pred_soft)
print("Confusion Matrix (Voting Classifier):")
print(cm)
print("\nClassification Report (Voting Classifier):")
print(classification_report(y_test, ensemble_pred_soft, target_names=['Low Life Expectancy', 'High Life Expectancy']))

# Visualize Confusion Matrix
fig_cm = go.Figure(data=go.Heatmap(z=cm,
                                   x=['Low', 'High'],
                                   y=['Low', 'High'],
                                   text=cm,
                                   texttemplate='%{text}',
                                   colorscale='Blues'))
fig_cm.update_layout(title='Confusion Matrix - Voting Classifier Ensemble',
                     xaxis_title='Predicted',
                     yaxis_title='True',
                     template='plotly_dark',
                     height=500)
fig_cm.write_html('../viz/confusion_matrix_ensemble.html')
fig_cm.show()

# Model Comparison Visualization
models = ['Random Forest', 'Gradient Boosting', 'Logistic Regression', 'KNN', 'Voting Ensemble']
accuracies = [rf_acc, gb_acc, lr_acc, knn_acc, ensemble_acc_soft]

fig_comparison = px.bar(x=models, y=accuracies, template='plotly_dark',
                        title='Model Performance Comparison',
                        labels={'x': 'Model', 'y': 'Accuracy'},
                        color=accuracies,
                        color_continuous_scale='Viridis')
fig_comparison.update_layout(height=500, showlegend=False)
fig_comparison.write_html('../viz/model_comparison.html')
fig_comparison.show()

Confusion Matrix (Voting Classifier):
[[1936  269]
 [ 175 2030]]

Classification Report (Voting Classifier):
                      precision    recall  f1-score   support

 Low Life Expectancy       0.92      0.88      0.90      2205
High Life Expectancy       0.88      0.92      0.90      2205

            accuracy                           0.90      4410
           macro avg       0.90      0.90      0.90      4410
        weighted avg       0.90      0.90      0.90      4410



In [15]:
# Feature Importance from Random Forest (best individual model)
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top Features by Importance:")
print(feature_importance_df.head(10))

# Visualize Feature Importance
fig_importance = px.bar(feature_importance_df.head(12), 
                        x='Importance', y='Feature',
                        template='plotly_dark',
                        title='Top 12 Features by Importance (Random Forest)',
                        color='Importance',
                        color_continuous_scale='Plasma',
                        orientation='h')
fig_importance.update_layout(height=600)
fig_importance.write_html('../viz/feature_importance.html')
fig_importance.show()

Top Features by Importance:
                   Feature  Importance
1    Infant Mortality Rate    0.231915
2   Under 5 Mortality Rate    0.213249
4               Birth Rate    0.141618
6           GDP per Capita    0.119635
11          Gender_encoded    0.095580
5               Death Rate    0.078907
7         Total Population    0.043570
0                     Year    0.027809
8                  Doctors    0.022777
9      Nurses and Midwifes    0.018564


## Section 9: Create Prediction Pipeline

In [16]:
# Create a comprehensive pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ensemble', voting_clf_soft)
])

# Re-fit the entire pipeline on scaled data for consistency
pipeline.fit(X_train, y_train)

# Test pipeline
pipeline_pred = pipeline.predict(X_test)
pipeline_acc = accuracy_score(y_test, pipeline_pred)
print(f"Pipeline Accuracy on Test Set: {pipeline_acc:.4f}")

# Save the pipeline
joblib.dump(pipeline, '../models/health_prediction_ensemble_pipeline.joblib')
print("Pipeline saved to ../models/health_prediction_ensemble_pipeline.joblib")

# Save the scaler separately for reference
joblib.dump(scaler, '../models/health_prediction_scaler.joblib')

# Save the label encoder for gender
joblib.dump(le_gender, '../models/health_prediction_gender_encoder.joblib')

# Save feature names
with open('../models/health_prediction_feature_names.txt', 'w') as f:
    f.write(','.join(X.columns))

print("All model artifacts saved successfully!")

Pipeline Accuracy on Test Set: 0.8993
Pipeline saved to ../models/health_prediction_ensemble_pipeline.joblib
All model artifacts saved successfully!


## Section 10: Example Predictions with Pipeline

In [18]:
# Example 1: Prediction for a developed country scenario
sample_1 = pd.DataFrame({
    'Year': [2020],
    'Infant Mortality Rate': [3.0],
    'Under 5 Mortality Rate': [4.0],
    'Suicides Rate': [15.0],
    'Birth Rate': [11.0],
    'Death Rate': [9.0],
    'GDP per Capita': [45000.0],
    'Total Population': [330000000.0],
    'Doctors': [300.0],
    'Nurses and Midwifes': [800.0],
    'Road Traffic Deaths': [12.0],
    'Gender_encoded': [1]  # Female
})

pred_1 = pipeline.predict(sample_1)
pred_1_proba = pipeline.predict_proba(sample_1)

print("Example 1 - Developed Country Scenario (2020):")
print(f"Prediction: {'High Life Expectancy' if pred_1[0] == 1 else 'Low Life Expectancy'}")
print(f"Confidence: {max(pred_1_proba[0]) * 100:.2f}%")
print(f"Probabilities: {pred_1_proba[0]}")

# Example 2: Prediction for a developing country scenario
sample_2 = pd.DataFrame({
    'Year': [2020],
    'Infant Mortality Rate': [35.0],
    'Under 5 Mortality Rate': [45.0],
    'Suicides Rate': [20.0],
    'Birth Rate': [35.0],
    'Death Rate': [8.0],
    'GDP per Capita': [3000.0],
    'Total Population': [1000000000.0],
    'Doctors': [20.0],
    'Nurses and Midwifes': [50.0],
    'Road Traffic Deaths': [25.0],
    'Gender_encoded': [0]  # Male
})

pred_2 = pipeline.predict(sample_2)
pred_2_proba = pipeline.predict_proba(sample_2)

print("\nExample 2 - Developing Country Scenario (2020):")
print(f"Prediction: {'High Life Expectancy' if pred_2[0] == 1 else 'Low Life Expectancy'}")
print(f"Confidence: {max(pred_2_proba[0]) * 100:.2f}%")
print(f"Probabilities: {pred_2_proba[0]}")

# Example 3: Prediction for emerging market scenario
sample_3 = pd.DataFrame({
    'Year': [2020],
    'Infant Mortality Rate': [15.0],
    'Under 5 Mortality Rate': [18.0],
    'Suicides Rate': [12.0],
    'Birth Rate': [18.0],
    'Death Rate': [7.0],
    'GDP per Capita': [10000.0],
    'Total Population': [300000000.0],
    'Doctors': [150.0],
    'Nurses and Midwifes': [400.0],
    'Road Traffic Deaths': [18.0],
    'Gender_encoded': [1]  # Female
})

pred_3 = pipeline.predict(sample_3)
pred_3_proba = pipeline.predict_proba(sample_3)

print("\nExample 3 - Emerging Market Scenario (2020):")
print(f"Prediction: {'High Life Expectancy' if pred_3[0] == 1 else 'Low Life Expectancy'}")
print(f"Confidence: {max(pred_3_proba[0]) * 100:.2f}%")
print(f"Probabilities: {pred_3_proba[0]}")

print("\n" + "="*60)
print("Pipeline ready for deployment and predictions!")
print("="*60)

Example 1 - Developed Country Scenario (2020):
Prediction: High Life Expectancy
Confidence: 98.94%
Probabilities: [0.01058554 0.98941446]

Example 2 - Developing Country Scenario (2020):
Prediction: Low Life Expectancy
Confidence: 98.60%
Probabilities: [0.98600067 0.01399933]

Example 3 - Emerging Market Scenario (2020):
Prediction: High Life Expectancy
Confidence: 94.70%
Probabilities: [0.05299096 0.94700904]

Pipeline ready for deployment and predictions!
